In [0]:
from pyspark.sql import functions as F

bronze_orders_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "bronze/orders/"
)

silver_orders_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "silver/orders/"
)


# Read Bronze
bronze_orders_df = (
    spark.read
    .format("delta")
    .load(bronze_orders_path)
)


# Read Silver
silver_orders_df = (
    spark.read
    .format("delta")
    .load(silver_orders_path)
)


# Counts
bronze_count = bronze_orders_df.count()
silver_count = silver_orders_df.count()


print("Bronze Orders:", bronze_count)
print("Silver Orders:", silver_count)

Bronze Orders: 235000
Silver Orders: 55968


In [0]:
difference = bronze_count - silver_count

difference_percentage = (
    difference / bronze_count
) * 100

print("Bronze records:", bronze_count)
print("Silver records:", silver_count)
print("Difference:", difference)
print(
    "Difference percentage:",
    round(difference_percentage, 2),
    "%"
)

Bronze records: 235000
Silver records: 55968
Difference: 179032
Difference percentage: 76.18 %


In [0]:
bronze_order_ids = (
    bronze_orders_df
    .select("order_id")
    .distinct()
)

silver_order_ids = (
    silver_orders_df
    .select("order_id")
    .distinct()
)

missing_orders_df = (
    bronze_order_ids
    .join(
        silver_order_ids,
        on="order_id",
        how="left_anti"
    )
)

missing_count = missing_orders_df.count()

print("Bronze order IDs missing from Silver:", missing_count)

display(
    missing_orders_df
    .orderBy("order_id")
    .limit(20)
)

Bronze order IDs missing from Silver: 12007


order_id
null
500003
500008
500013
500018
500023
500028
500033
500038
500043


In [0]:
missing_orders_detail = (
    bronze_orders_df
    .join(
        missing_orders_df,
        on="order_id",
        how="inner"
    )
)

display(
    missing_orders_detail
    .groupBy("source_file", "batch_id")
    .agg(
        F.count("*").alias("missing_count"),
        F.min("ingestion_timestamp").alias("min_ingestion"),
        F.max("ingestion_timestamp").alias("max_ingestion")
    )
    .orderBy(F.desc("missing_count"))
)

source_file,batch_id,missing_count,min_ingestion,max_ingestion
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=06/orders_20261006.json,B202609301615,40000,2026-10-06T13:04:22.940Z,2026-10-06T15:04:50.100Z
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=04/orders_20261004.json,B202609301615,5000,2026-10-04T13:38:12.300Z,2026-10-04T13:38:12.300Z
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=03/orders_20261003.json,B202609301615,2006,2026-10-03T15:44:03.772Z,2026-10-03T15:44:03.772Z


In [0]:
from pyspark.sql import functions as F

control_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/pipeline_watermark/"
)

watermark_df = (
    spark.read
    .format("delta")
    .load(control_path)
)

display(
    watermark_df
    .orderBy("pipeline_name")
)

pipeline_name,last_processed_timestamp,last_run_id,updated_at
customers_silver,2026-10-06T15:04:39.407Z,"CUSTOMERS_REBUILD_Column<'date_format(current_timestamp(), yyyyMMddHHmmss)'>",2026-10-06T15:05:01.849Z
gold_customer_sales,2026-10-06T13:04:22.940Z,GOLD_CUSTOMER_SALES_RUN,2026-10-06T14:04:56.032Z
gold_daily_sales,2026-10-06T13:04:22.940Z,GOLD_DAILY_SALES_20261006130422,2026-10-06T14:45:34.819Z
gold_market_sales,2026-10-06T13:04:22.940Z,GOLD_MARKET_SALES_RUN,2026-10-06T14:56:07.984Z
gold_product_sales,2026-10-06T13:04:22.940Z,GOLD_PRODUCT_SALES_20261006130422,2026-10-06T15:01:30.672Z
orders_silver,2026-10-06T21:00:39.000Z,RUN_117da8d5,2026-10-06T13:04:55.952Z
products_silver,2026-10-06T15:04:35.750Z,null,2026-10-06T15:05:04.080Z
